# Phase 4 — Lapse Analysis: Where Do Lost Shoppers Go?

**Purpose:** Identify when and why shoppers lapse from アリエールジェル, and track their destination at sub-brand + size level.

| Step | Description |
|------|-------------|
| 4-1 | Identify lapse events: 6+ months since last purchase |
| 4-2 | Lapse rate by size and ASP band |
| 4-3 | Post-lapse destination — sub-brand AND size level |

**Lapse Definition:** Shopper whose last アリエールジェル purchase has no follow-up within 6 months.

**Post-Lapse Destinations:**
- Within P&G laundry (other sub-brands) — by sub-brand + size
- To アタック抗菌EX — by size
- To other laundry brands — by sub-brand + size
- Category exit (no laundry purchase at all)

**Created:** 2026-02-19

---
## 0. Imports & Connection

In [1]:
import os
import pandas as pd
import numpy as np
import warnings
from dotenv import load_dotenv
import databricks.sql as sql
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

warnings.filterwarnings('ignore')
pd.set_option('display.max_rows', 200)
pd.set_option('display.float_format', lambda x: f'{x:,.1f}')

def _find_japanese_font():
    for name in ['MS Gothic', 'MS PGothic', 'Yu Gothic', 'Meiryo', 'IPAexGothic']:
        if name in {f.name for f in fm.fontManager.ttflist}:
            return name
    return None
_jp_font = _find_japanese_font()
if _jp_font:
    plt.rcParams['font.family'] = _jp_font
    print(f'✅ Japanese font: {_jp_font}')

load_dotenv(dotenv_path='../../.env')
DATABRICKS_HOST      = os.getenv('DATABRICKS_HOST')
DATABRICKS_TOKEN     = os.getenv('DATABRICKS_TOKEN')
DATABRICKS_HTTP_PATH = os.getenv('DATABRICKS_HTTP_PATH')
assert all([DATABRICKS_HOST, DATABRICKS_TOKEN, DATABRICKS_HTTP_PATH]), 'Missing .env credentials'
print('✅ Credentials loaded')

def execute_query(query: str) -> pd.DataFrame:
    with sql.connect(server_hostname=DATABRICKS_HOST, http_path=DATABRICKS_HTTP_PATH,
                     access_token=DATABRICKS_TOKEN) as conn:
        with conn.cursor() as cur:
            cur.execute(query)
            result = cur.fetchall()
            columns = [d[0] for d in cur.description]
            return pd.DataFrame(result, columns=columns)

✅ Japanese font: MS Gothic
✅ Credentials loaded


---
## 1. Parameters

In [ ]:
ARIEL_GEL   = 'ｱﾘｴｰﾙｼﾞｪﾙ'
ATTACK_EX   = 'ｱﾀｯｸ抗菌EX'
SUB_CAT     = '洗濯洗剤'
CATEGORY    = 'Laundry'

ANALYSIS_START = '2025-01-01'
ANALYSIS_END   = '2026-01-31'
RENEWAL_MONTH  = '2025-05-01'

# ── Canonical parameters (shared across NB00–NB05) ───────────────────
TRIAL_LOOKBACK_DAYS      = 365   # 12-month lookback for trial definition
REPEAT_LAPSE_WINDOW_DAYS = 180   # 6-month repeat / lapse window

# Lapse = 6 months with no return (alias for backward compat)
LAPSE_WINDOW_DAYS = REPEAT_LAPSE_WINDOW_DAYS

# To identify lapse, we need shoppers whose last purchase is at least 6 months before data end
LAPSE_CUTOFF_DATE = '2025-07-31'  # Last purchase before this date = 6 months to confirm lapse

RETAILER_CODES = [
    'cds_8005', 'cds_8006', 'cds_8007', 'cds_8008', 'cds_8009',
    'cds_8010', 'cds_8011', 'cds_8012', 'cds_8013',
]
RETAILER_IN = ', '.join(f"'{c}'" for c in RETAILER_CODES)

# ── Size order (physical size: small → large) and exclusions ──────────
SIZE_ORDER     = ['本体通常', '詰替超特大', '詰替ｳﾙﾄﾗｼﾞｬﾝﾎﾞ', '詰替超ｳﾙﾄﾗｼﾞｬﾝﾎﾞ', '詰替ﾒｶﾞｼﾞｬﾝﾎﾞ']
EXCLUDED_SIZES = ['ｿﾉﾀ', '詰替通常', '詰替超ｼﾞｬﾝﾎﾞ']

def order_and_filter_sizes(sizes_list):
    """Return sizes in SIZE_ORDER, excluding EXCLUDED_SIZES."""
    sizes_set = set(sizes_list) - set(EXCLUDED_SIZES)
    return [s for s in SIZE_ORDER if s in sizes_set]

print(f'📋 Lapse definition: no return within {LAPSE_WINDOW_DAYS} days')
print(f'📋 Lapse cutoff: last Ariel purchase before {LAPSE_CUTOFF_DATE}')
print(f'📋 Post-lapse tracking window: up to {ANALYSIS_END}')
print(f'📋 Size order: {SIZE_ORDER}')


📋 Lapse definition: no return within 180 days
📋 Lapse cutoff: last Ariel purchase before 2025-07-31
📋 Post-lapse tracking window: up to 2026-01-31
📋 Size order: ['本体通常', '詰替超特大', '詰替ｳﾙﾄﾗｼﾞｬﾝﾎﾞ', '詰替超ｳﾙﾄﾗｼﾞｬﾝﾎﾞ', '詰替ﾒｶﾞｼﾞｬﾝﾎﾞ']


### Canonical Definitions (shared across NB00–NB05)

| Concept | Rule | Parameter |
|---------|------|-----------|
| **Trial shopper** | No purchase of the sub-brand in the prior **12 months (365 days)** — detected via `LAG()` window function | `TRIAL_LOOKBACK_DAYS = 365` |
| **Repeat shopper** | ≥1 additional purchase within **180 days** of trial | `REPEAT_LAPSE_WINDOW_DAYS = 180` |
| **Lapsed shopper** | Zero purchases within **180 days** of last purchase | `REPEAT_LAPSE_WINDOW_DAYS = 180` |
| **ASP bin** | `FLOOR(ASP / 50) * 50` — 50 JPY floor bins | x-axis: "ASP (50 JPY bin)" |
| **ASP threshold (NB04)** | `MIN_FREQ = 30` shoppers AND `MIN_WEEK_STORE = 5` week×store — **INTENTIONAL EXCEPTION** for lapse rate robustness (higher threshold than NB02/NB03 because lapse rate is a ratio sensitive to small denominators) | See code cell |

> **Note:** NB04 identifies lapse from a shopper's *last purchase date* (not from a trial event), so the 12-month trial lookback does not apply to lapse identification queries.

---
## 2. Step 4-1: Identify Lapse Events

In [3]:
# ── Identify shoppers who lapsed from Ariel Gel ───────────────────────
# A shopper lapses if their last Ariel Gel purchase is 6+ months ago and
# they have NOT returned to Ariel Gel.

lapse_query = f"""
WITH ariel_purchases AS (
    SELECT
        idpos.shopper_key,
        prod.jp_segment_4_name AS size_code,
        CAST(idpos.sales_period_group_end_date_part AS DATE) AS purchase_date,
        SUM(idpos.pos_sales_amt) / SUM(idpos.pos_unit_sales_qty) AS asp
    FROM cdl_customer_prod.gold_customer_loyalty.loyalty_transact_fct_v1_vw idpos
    LEFT JOIN id_pos_ai_1.prod_dim_ext_vw prod
           ON idpos.prod_key = prod.prod_key
    LEFT JOIN id_pos_ai_1.shopper_dim_generic_vw shopper
           ON idpos.shopper_key = shopper.shopper_key
    WHERE idpos.sales_period_group_end_date_part BETWEEN '{ANALYSIS_START}' AND '{ANALYSIS_END}'
      AND idpos.data_provider_code_part IN ({RETAILER_IN})
      AND prod.jp_category_name = '{CATEGORY}'
      AND prod.jp_sub_category_alter_lang_name = '{SUB_CAT}'
      AND prod.jp_sub_brand_alter_lang_name = '{ARIEL_GEL}'
      AND shopper.member_ind = 'Y'
      AND idpos.pos_unit_sales_qty > 0
    GROUP BY 1, 2, 3
),
-- Find each shopper's last Ariel purchase and its size/ASP
last_ariel AS (
    SELECT
        shopper_key,
        MAX(purchase_date) AS last_ariel_date
    FROM ariel_purchases
    GROUP BY 1
    HAVING MAX(purchase_date) <= '{LAPSE_CUTOFF_DATE}'  -- Must be old enough to confirm lapse
),
-- Check if shopper returned to Ariel after their last purchase
lapse_check AS (
    SELECT
        la.shopper_key,
        la.last_ariel_date,
        -- Check for any Ariel purchase within 6 months after last_ariel_date
        MAX(CASE WHEN ap2.purchase_date > la.last_ariel_date
                  AND ap2.purchase_date <= DATE_ADD(la.last_ariel_date, {LAPSE_WINDOW_DAYS})
                 THEN 1 ELSE 0 END) AS returned
    FROM last_ariel la
    LEFT JOIN ariel_purchases ap2
           ON la.shopper_key = ap2.shopper_key
          AND ap2.purchase_date > la.last_ariel_date
    GROUP BY 1, 2
),
-- Filter to lapsed shoppers only
lapsed_shoppers AS (
    SELECT shopper_key, last_ariel_date
    FROM lapse_check
    WHERE returned = 0
)
-- Get the last Ariel purchase detail (size + ASP)
SELECT
    ls.shopper_key,
    ls.last_ariel_date,
    ap.size_code AS last_ariel_size,
    ap.asp       AS last_ariel_asp
FROM lapsed_shoppers ls
INNER JOIN ariel_purchases ap
       ON ls.shopper_key    = ap.shopper_key
      AND ls.last_ariel_date = ap.purchase_date
ORDER BY ls.last_ariel_date
"""

print('⏳ Identifying lapsed shoppers...', flush=True)
df_lapsed = execute_query(lapse_query)
df_lapsed['last_ariel_date'] = pd.to_datetime(df_lapsed['last_ariel_date'])
df_lapsed['last_ariel_asp'] = pd.to_numeric(df_lapsed['last_ariel_asp'])

print(f'\n✅ {len(df_lapsed):,} lapsed shoppers identified')
print(f'   Date range: {df_lapsed["last_ariel_date"].min().date()} → {df_lapsed["last_ariel_date"].max().date()}')

⏳ Identifying lapsed shoppers...


HTTP request failed after retries: HTTPSConnectionPool(host='https', port=443): Max retries exceeded with url: //adb-2258763851730787.7.azuredatabricks.net/api/2.0/connector-service/feature-flags/PYTHON/4.2.3 (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x000001CEC8A76170>: Failed to resolve 'https' ([Errno 11001] getaddrinfo failed)"))



✅ 1,809,811 lapsed shoppers identified
   Date range: 2025-01-01 → 2025-07-31


---
## 3. Step 4-2: Lapse Rate by Size and ASP Band

In [4]:
# ── Get total active shoppers per size for lapse rate calculation ──────
active_query = f"""
SELECT
    prod.jp_segment_4_name AS size_code,
    COUNT(DISTINCT idpos.shopper_key) AS active_shoppers
FROM cdl_customer_prod.gold_customer_loyalty.loyalty_transact_fct_v1_vw idpos
LEFT JOIN id_pos_ai_1.prod_dim_ext_vw prod
       ON idpos.prod_key = prod.prod_key
LEFT JOIN id_pos_ai_1.shopper_dim_generic_vw shopper
       ON idpos.shopper_key = shopper.shopper_key
WHERE idpos.sales_period_group_end_date_part BETWEEN '{ANALYSIS_START}' AND '{LAPSE_CUTOFF_DATE}'
  AND idpos.data_provider_code_part IN ({RETAILER_IN})
  AND prod.jp_category_name = '{CATEGORY}'
  AND prod.jp_sub_category_alter_lang_name = '{SUB_CAT}'
  AND prod.jp_sub_brand_alter_lang_name = '{ARIEL_GEL}'
  AND shopper.member_ind = 'Y'
  AND idpos.pos_unit_sales_qty > 0
GROUP BY 1
"""

print('⏳ Fetching active shoppers per size...', flush=True)
df_active = execute_query(active_query)
df_active['active_shoppers'] = pd.to_numeric(df_active['active_shoppers'])

# Apply exclusion filter
df_active = df_active[~df_active['size_code'].isin(EXCLUDED_SIZES)]

# Lapse count per size (exclude small sizes)
lapse_by_size = df_lapsed[~df_lapsed['last_ariel_size'].isin(EXCLUDED_SIZES)].groupby('last_ariel_size').agg(
    lapsed_shoppers=('shopper_key', 'nunique')
).reset_index().rename(columns={'last_ariel_size': 'size_code'})

# Merge
lapse_rate = lapse_by_size.merge(df_active, on='size_code', how='left')
lapse_rate['lapse_rate_%'] = (lapse_rate['lapsed_shoppers'] / lapse_rate['active_shoppers'] * 100).round(1)

# Reorder by SIZE_ORDER
lapse_rate['_sort'] = lapse_rate['size_code'].map(
    {s: i for i, s in enumerate(SIZE_ORDER)}
).fillna(99)
lapse_rate = lapse_rate.sort_values('_sort').drop(columns='_sort')

print('\n' + '=' * 60)
print('Lapse Rate by Size — アリエールジェル')
print('=' * 60)
print(lapse_rate.to_string(index=False))


⏳ Fetching active shoppers per size...


HTTP request error: 'NoneType' object has no attribute 'request'



Lapse Rate by Size — アリエールジェル
    size_code  lapsed_shoppers  active_shoppers  lapse_rate_%
         本体通常           406262           740575          54.9
        詰替超特大           494040          1108637          44.6
 詰替ｳﾙﾄﾗｼﾞｬﾝﾎﾞ           322201           819128          39.3
詰替超ｳﾙﾄﾗｼﾞｬﾝﾎﾞ           392207           918717          42.7
  詰替ﾒｶﾞｼﾞｬﾝﾎﾞ           175587           349646          50.2


HTTP request error: 'NoneType' object has no attribute 'request'
HTTP request error: 'NoneType' object has no attribute 'request'
HTTP request error: 'NoneType' object has no attribute 'request'
HTTP request error: 'NoneType' object has no attribute 'request'
HTTP request error: 'NoneType' object has no attribute 'request'
HTTP request error: 'NoneType' object has no attribute 'request'
HTTP request error: 'NoneType' object has no attribute 'request'
HTTP request error: 'NoneType' object has no attribute 'request'
HTTP request error: 'NoneType' object has no attribute 'request'
HTTP request error: 'NoneType' object has no attribute 'request'
HTTP request error: 'NoneType' object has no attribute 'request'


In [ ]:

# ── Lapse RATE by ASP band (50 JPY floor) per size ────────────────────
# ── BUG FIX: Scan ariel_purchases only up to LAPSE_CUTOFF_DATE ─────────
# Previous bug: scanning to ANALYSIS_END then HAVING MAX(date)<=cutoff
# excluded retained shoppers (their MAX date was the post-cutoff return date).
# Fix: narrow scan window to LAPSE_CUTOFF_DATE so all shoppers in the CTE
# have their last purchase naturally ≤ cutoff → correct lapse rate denominator.

MIN_FREQ       = 30   # INTENTIONAL EXCEPTION: higher than NB02 (2) / NB03 (5) because lapse rate
                      # is a ratio sensitive to small denominators — need robust base for credible %
MIN_WEEK_STORE = 5    # minimum (week × store) count per size×ASP band — removes noise price points

all_at_risk_query = f"""
WITH ariel_purchases AS (
    SELECT
        idpos.shopper_key,
        prod.jp_segment_4_name AS size_code,
        CAST(idpos.sales_period_group_end_date_part AS DATE) AS purchase_date,
        SUM(idpos.pos_sales_amt) / SUM(idpos.pos_unit_sales_qty) AS asp
    FROM cdl_customer_prod.gold_customer_loyalty.loyalty_transact_fct_v1_vw idpos
    LEFT JOIN id_pos_ai_1.prod_dim_ext_vw prod ON idpos.prod_key = prod.prod_key
    LEFT JOIN id_pos_ai_1.shopper_dim_generic_vw shopper ON idpos.shopper_key = shopper.shopper_key
    WHERE idpos.sales_period_group_end_date_part BETWEEN '{ANALYSIS_START}' AND '{LAPSE_CUTOFF_DATE}'
      AND idpos.data_provider_code_part IN ({RETAILER_IN})
      AND prod.jp_category_name = '{CATEGORY}'
      AND prod.jp_sub_category_alter_lang_name = '{SUB_CAT}'
      AND prod.jp_sub_brand_alter_lang_name = '{ARIEL_GEL}'
      AND shopper.member_ind = 'Y'
      AND idpos.pos_unit_sales_qty > 0
    GROUP BY 1, 2, 3
),
last_ariel AS (
    -- With scan window ending at LAPSE_CUTOFF_DATE, MAX(date) is naturally <= cutoff
    -- No HAVING needed (removed to fix the 100% lapse rate bug)
    SELECT shopper_key, MAX(purchase_date) AS last_ariel_date
    FROM ariel_purchases
    GROUP BY 1
)
SELECT la.shopper_key, la.last_ariel_date, ap.size_code, ap.asp
FROM last_ariel la
INNER JOIN ariel_purchases ap
       ON la.shopper_key = ap.shopper_key
      AND la.last_ariel_date = ap.purchase_date
"""

print('⏳ Fetching all at-risk shoppers with ASP (BUG FIX: scan to LAPSE_CUTOFF_DATE)...', flush=True)
df_all_at_risk = execute_query(all_at_risk_query)
df_all_at_risk['asp'] = pd.to_numeric(df_all_at_risk['asp'])
print(f'✅ {len(df_all_at_risk):,} at-risk shoppers fetched')

# Mark lapsed vs retained
lapsed_ids = set(df_lapsed['shopper_key'].unique())
df_all_at_risk['is_lapsed'] = df_all_at_risk['shopper_key'].isin(lapsed_ids).astype(int)
print(f'   Lapsed: {df_all_at_risk["is_lapsed"].sum():,}  Retained: {(~df_all_at_risk["is_lapsed"].astype(bool)).sum():,}')

# ── Week × store coverage per SIZE × ASP band ──────────────────────────
# Groups by size_code so each size's price coverage is evaluated independently.
# A price band that appears widely for 詰替超特大 but rarely for 本体通常
# will only be filtered out for 本体通常, keeping the 詰替超特大 data intact.
week_store_query = f"""
WITH store_week_asp AS (
    SELECT
        prod.jp_segment_4_name                AS size_code,
        idpos.site_key,
        idpos.sales_period_group_end_date_part AS week_dt,
        FLOOR(SUM(idpos.pos_sales_amt) / SUM(idpos.pos_unit_sales_qty) / 50) * 50
            AS asp_band
    FROM cdl_customer_prod.gold_customer_loyalty.loyalty_transact_fct_v1_vw idpos
    INNER JOIN id_pos_ai_1.prod_dim_ext_vw prod ON idpos.prod_key = prod.prod_key
    WHERE idpos.sales_period_group_end_date_part BETWEEN '{ANALYSIS_START}' AND '{LAPSE_CUTOFF_DATE}'
      AND idpos.data_provider_code_part IN ({RETAILER_IN})
      AND prod.jp_category_name = '{CATEGORY}'
      AND prod.jp_sub_category_alter_lang_name = '{SUB_CAT}'
      AND prod.jp_sub_brand_alter_lang_name = '{ARIEL_GEL}'
      AND idpos.pos_unit_sales_qty > 0
    GROUP BY 1, 2, 3
)
SELECT size_code, asp_band, COUNT(*) AS week_store_count
FROM store_week_asp
GROUP BY size_code, asp_band
ORDER BY size_code, asp_band
"""

print('⏳ Fetching week × store coverage per size × ASP band...', flush=True)
df_week_store = execute_query(week_store_query)
df_week_store['asp_band']         = pd.to_numeric(df_week_store['asp_band']).astype(int)
df_week_store['week_store_count'] = pd.to_numeric(df_week_store['week_store_count']).astype(int)
print(f'✅ {len(df_week_store)} (size × ASP band) rows found')

# All-sizes rollup: sum week_store_count across sizes per asp_band
df_week_store_all = df_week_store.groupby('asp_band')['week_store_count'].sum().reset_index()

print('\nWeek × store coverage per size:')
for sz in order_and_filter_sizes(df_week_store['size_code'].unique()):
    sub = df_week_store[df_week_store['size_code'] == sz]
    print(f'\n▶ {sz}:')
    print(sub[['asp_band', 'week_store_count']].to_string(index=False))

# Apply size exclusion
df_at_risk_filtered = df_all_at_risk[~df_all_at_risk['size_code'].isin(EXCLUDED_SIZES)].copy()

# Bin ASP by 50 JPY floor
df_at_risk_filtered['asp_band'] = (df_at_risk_filtered['asp'] // 50 * 50).astype(int)

# Compute lapse rate by ASP band + size
asp_lapse = df_at_risk_filtered.groupby(['size_code', 'asp_band']).agg(
    total_shoppers=('shopper_key', 'nunique'),
    lapsed_shoppers=('is_lapsed', 'sum')
).reset_index()
asp_lapse['lapse_rate_%'] = (asp_lapse['lapsed_shoppers'] / asp_lapse['total_shoppers'] * 100).round(1)

# Dual filter per size: merge on (size_code, asp_band) so each size uses its own coverage
asp_lapse = asp_lapse.merge(df_week_store[['size_code', 'asp_band', 'week_store_count']],
                            on=['size_code', 'asp_band'], how='left')
asp_lapse['week_store_count'] = asp_lapse['week_store_count'].fillna(0).astype(int)
asp_lapse_plot = asp_lapse[
    (asp_lapse['total_shoppers'] >= MIN_FREQ) &
    (asp_lapse['week_store_count'] >= MIN_WEEK_STORE)
].copy()

# Also compute ALL sizes combined (after exclusion)
asp_lapse_all = df_at_risk_filtered.groupby('asp_band').agg(
    total_shoppers=('shopper_key', 'nunique'),
    lapsed_shoppers=('is_lapsed', 'sum')
).reset_index()
asp_lapse_all['lapse_rate_%'] = (asp_lapse_all['lapsed_shoppers'] / asp_lapse_all['total_shoppers'] * 100).round(1)
asp_lapse_all['size_code'] = '(All Sizes)'
asp_lapse_all = asp_lapse_all[['size_code', 'asp_band', 'total_shoppers', 'lapsed_shoppers', 'lapse_rate_%']]
# For all-sizes panel use the summed coverage across sizes
asp_lapse_all = asp_lapse_all.merge(df_week_store_all[['asp_band', 'week_store_count']],
                                    on='asp_band', how='left')
asp_lapse_all['week_store_count'] = asp_lapse_all['week_store_count'].fillna(0).astype(int)
asp_lapse_all_plot = asp_lapse_all[
    (asp_lapse_all['total_shoppers'] >= MIN_FREQ) &
    (asp_lapse_all['week_store_count'] >= MIN_WEEK_STORE)
].copy()

print(f'\nDual filter: shoppers >= {MIN_FREQ} AND week×store coverage >= {MIN_WEEK_STORE} (per size)')
print(f'  Bands kept (All Sizes):   {len(asp_lapse_all_plot)} / {len(asp_lapse_all)}')
print(f'  Bands kept (Per Size):    {len(asp_lapse_plot)} / {len(asp_lapse)}')

print('\nLapse Rate by 50 JPY ASP Band — All Sizes Combined:')
print('=' * 70)
print(asp_lapse_all_plot.to_string(index=False))

print('\nLapse Rate by 50 JPY ASP Band — Per Size:')
print('=' * 70)
for size in order_and_filter_sizes(asp_lapse_plot['size_code'].unique()):
    s = asp_lapse_plot[asp_lapse_plot['size_code'] == size]
    print(f'\n▶ {size}:')
    print(s.to_string(index=False))

# ── Visualization: Lapse RATE by 50 JPY ASP band ─────────────────────
asp_lapse_combined = pd.concat([asp_lapse_all_plot, asp_lapse_plot], ignore_index=True)
plot_sizes = ['(All Sizes)'] + order_and_filter_sizes(asp_lapse_plot['size_code'].unique())

n_s = len(plot_sizes)
n_c = min(2, n_s)
n_r = (n_s + n_c - 1) // n_c

fig = make_subplots(rows=n_r, cols=n_c,
                    subplot_titles=[f'{s}: Lapse Rate by ASP Band' for s in plot_sizes],
                    vertical_spacing=0.10)

for idx, size in enumerate(plot_sizes):
    row = idx // n_c + 1
    col = idx % n_c + 1
    subset = asp_lapse_combined[asp_lapse_combined['size_code'] == size].sort_values('asp_band')
    if len(subset) == 0:
        continue
    fig.add_trace(go.Bar(
        x=subset['asp_band'].astype(str) + '~',
        y=subset['lapse_rate_%'],
        name=size, marker_color='#FF6B6B',
        text=[f'{r:.0f}%<br>({int(l)}/{int(t)})<br>wk×st={int(w)}' for r, l, t, w in
              zip(subset['lapse_rate_%'], subset['lapsed_shoppers'], subset['total_shoppers'], subset['week_store_count'])],
        textposition='outside', showlegend=False),
        row=row, col=col)
    fig.update_xaxes(title_text='ASP (50 JPY bin)', row=row, col=col)
    fig.update_yaxes(title_text='Lapse Rate (%)', row=row, col=col)

fig.update_layout(height=350*n_r,
                  title_text=f'At Which ASP Does Lapse Rate Peak? (50 JPY bins, shoppers≥{MIN_FREQ}, wk×store≥{MIN_WEEK_STORE} per size)',
                  template='plotly_white')
fig.show()


⏳ Fetching all at-risk shoppers with ASP (BUG FIX: scan to LAPSE_CUTOFF_DATE)...


✅ 3,169,432 at-risk shoppers fetched
   Lapsed: 1,809,811  Retained: 1,359,621
⏳ Fetching week × store coverage per size × ASP band...
✅ 190 (size × ASP band) rows found

Week × store coverage per size:

▶ 本体通常:
 asp_band  week_store_count
        0              1900
       50               934
      100              2413
      150            133569
      200             38280
      250             55359
      300              7131
      350             33363
      400             14370
      450             55409
      500             31112
      550             30661
      600              3528
      650                 7
      700                 3
      750                11
      800                 2
      900                 2
     1000                 2

▶ 詰替超特大:
 asp_band  week_store_count
        0              2892
       50               667
      100              1811
      150              6355
      200             14033
      250            157656
      300            2

---
## 4. Step 4-3: Post-Lapse Destination — Sub-brand + Size Level

For each lapsed アリエールジェル shopper, find their **next laundry purchase** after leaving Ariel, classified by:
- Sub-brand (ｱﾀｯｸ抗菌EX, ﾅﾉｯｸｽﾜﾝ, ﾎﾞｰﾙﾄﾞｼﾞｪﾙ, etc.)
- Size (same `jp_segment_4_name`)
- Or: no next purchase = category exit

In [6]:
# ── Find next laundry purchase after lapse for each shopper ───────────
# OPTIMIZED: Scan Ariel purchases through ANALYSIS_END (full window).
# HAVING MAX(date) <= LAPSE_CUTOFF_DATE guarantees confirmed lapse
# without the expensive NOT EXISTS correlated subquery.

destination_query = f"""
WITH ariel_last AS (
    -- Last Ariel purchase per shopper across the FULL analysis window
    -- HAVING ensures last purchase was before cutoff → confirmed lapsed
    SELECT
        idpos.shopper_key,
        MAX(CAST(idpos.sales_period_group_end_date_part AS DATE)) AS last_ariel_date
    FROM cdl_customer_prod.gold_customer_loyalty.loyalty_transact_fct_v1_vw idpos
    INNER JOIN id_pos_ai_1.prod_dim_ext_vw prod ON idpos.prod_key = prod.prod_key
    INNER JOIN id_pos_ai_1.shopper_dim_generic_vw shopper ON idpos.shopper_key = shopper.shopper_key
    WHERE idpos.sales_period_group_end_date_part BETWEEN '{ANALYSIS_START}' AND '{ANALYSIS_END}'
      AND idpos.data_provider_code_part IN ({RETAILER_IN})
      AND prod.jp_category_name = '{CATEGORY}'
      AND prod.jp_sub_category_alter_lang_name = '{SUB_CAT}'
      AND prod.jp_sub_brand_alter_lang_name = '{ARIEL_GEL}'
      AND shopper.member_ind = 'Y'
      AND idpos.pos_unit_sales_qty > 0
    GROUP BY 1
    HAVING MAX(CAST(idpos.sales_period_group_end_date_part AS DATE)) <= DATE('{LAPSE_CUTOFF_DATE}')
),
-- Find their FIRST next purchase in laundry liquid (any brand/sub-brand)
next_laundry AS (
    SELECT
        al.shopper_key,
        prod.jp_sub_brand_alter_lang_name   AS next_sub_brand,
        prod.jp_segment_4_name              AS next_size,
        ROW_NUMBER() OVER (
            PARTITION BY al.shopper_key
            ORDER BY CAST(idpos.sales_period_group_end_date_part AS DATE)
        ) AS rn
    FROM ariel_last al
    INNER JOIN cdl_customer_prod.gold_customer_loyalty.loyalty_transact_fct_v1_vw idpos
           ON al.shopper_key = idpos.shopper_key
    INNER JOIN id_pos_ai_1.prod_dim_ext_vw prod
           ON idpos.prod_key = prod.prod_key
    WHERE CAST(idpos.sales_period_group_end_date_part AS DATE) > al.last_ariel_date
      AND idpos.sales_period_group_end_date_part <= '{ANALYSIS_END}'
      AND idpos.data_provider_code_part IN ({RETAILER_IN})
      AND prod.jp_category_name = '{CATEGORY}'
      AND prod.jp_sub_category_alter_lang_name = '{SUB_CAT}'
      AND idpos.pos_unit_sales_qty > 0
)
SELECT
    shopper_key,
    next_sub_brand,
    next_size
FROM next_laundry
WHERE rn = 1
"""

print('⏳ Tracking post-lapse destinations (optimized query, may take several minutes)...', flush=True)
df_destination = execute_query(destination_query)

print(f'\n✅ {len(df_destination):,} lapsed shoppers with next purchase tracked')
print(f'   (Out of {df_lapsed["shopper_key"].nunique():,} total lapsed shoppers)')

⏳ Tracking post-lapse destinations (optimized query, may take several minutes)...

✅ 892,873 lapsed shoppers with next purchase tracked
   (Out of 1,783,582 total lapsed shoppers)


In [7]:
# ── Join back lapsed shopper's last Ariel size ────────────────────────
df_dest_full = df_destination.merge(
    df_lapsed[['shopper_key', 'last_ariel_size', 'last_ariel_asp']],
    on='shopper_key', how='left'
)

# ── Summary: post-lapse destination at sub-brand + size level ─────────
dest_summary = df_dest_full.groupby(['next_sub_brand', 'next_size']).agg(
    shoppers=('shopper_key', 'nunique')
).reset_index().sort_values('shoppers', ascending=False)

total_tracked = dest_summary['shoppers'].sum()
total_lapsed  = df_lapsed['shopper_key'].nunique()
category_exit = total_lapsed - total_tracked

print('=' * 80)
print('Post-Lapse Destination: Where Did Lapsed アリエールジェル Shoppers Go?')
print(f'Total lapsed: {total_lapsed:,} | Tracked next purchase: {total_tracked:,} | Category exit: {category_exit:,}')
print('=' * 80)

dest_summary['share_%'] = (dest_summary['shoppers'] / total_lapsed * 100).round(1)
print(dest_summary.head(20).to_string(index=False))

print(f'\n📊 Category Exit (no laundry purchase after lapse): {category_exit:,} ({category_exit/total_lapsed*100:.1f}%)')

Post-Lapse Destination: Where Did Lapsed アリエールジェル Shoppers Go?
Total lapsed: 1,783,582 | Tracked next purchase: 892,873 | Category exit: 890,709
next_sub_brand     next_size  shoppers  share_%
      ｱﾀｯｸ抗菌EX 詰替超ｳﾙﾄﾗｼﾞｬﾝﾎﾞ    141498      7.9
      ｱﾀｯｸ抗菌EX         詰替超特大     87689      4.9
      ｱﾀｯｸ抗菌EX  詰替ｳﾙﾄﾗｼﾞｬﾝﾎﾞ     81855      4.6
      ｱﾀｯｸ抗菌EX   詰替ﾒｶﾞｼﾞｬﾝﾎﾞ     58705      3.3
      ｱﾀｯｸ抗菌EX          本体通常     43455      2.4
 ﾆｭｰﾋﾞｰｽﾞ ｼﾞｪﾙ         詰替超特大     23789      1.3
    ﾎﾞｰﾙﾄﾞｼﾞｪﾙ          本体通常     22460      1.3
          ｴﾏｰﾙ         詰替超特大     22069      1.2
    ﾎﾞｰﾙﾄﾞｼﾞｪﾙ         詰替超特大     20981      1.2
ﾎﾞｰﾙﾄﾞｼﾞｪﾙﾎﾞｰﾙ          本体通常     20240      1.1
          ｱｸﾛﾝ         詰替超特大     17059      1.0
 ｱﾘｴｰﾙｼﾞｪﾙﾎﾞｰﾙ 詰替ﾊｲﾊﾟｰｼﾞｬﾝﾎﾞ     14408      0.8
        ｱﾀｯｸ粉末          詰替通常     13672      0.8
   ﾆｭｰﾋﾞｰｽﾞ 粉末          本体通常     13343      0.7
           ｿﾉﾀ           ｿﾉﾀ     13314      0.7
 ﾆｭｰﾋﾞｰｽﾞ ｼﾞｪﾙ  詰替ｳﾙﾄﾗｼﾞｬﾝﾎﾞ     12603      0.7
      ｱﾀｯｸZERO     詰替超ｼﾞｬﾝﾎﾞ     11875 

In [8]:
# ── Breakdown by last Ariel size → destination sub-brand + size ───────
flow_detail = df_dest_full.groupby(['last_ariel_size', 'next_sub_brand', 'next_size']).agg(
    shoppers=('shopper_key', 'nunique')
).reset_index().sort_values(['last_ariel_size', 'shoppers'], ascending=[True, False])

print('\nLapse Flow Detail: Last Ariel Size → Next Sub-brand + Size')
print('=' * 80)

for size in order_and_filter_sizes(list(flow_detail['last_ariel_size'].unique())):
    size_data = flow_detail[flow_detail['last_ariel_size'] == size]
    total_from_size = size_data['shoppers'].sum()
    size_data_pct = size_data.copy()
    size_data_pct['share_%'] = (size_data_pct['shoppers'] / total_from_size * 100).round(1)
    print(f'\n▶ From {size} ({total_from_size:,} shoppers):')
    print(size_data_pct.head(10).to_string(index=False))


Lapse Flow Detail: Last Ariel Size → Next Sub-brand + Size

▶ From 本体通常 (223,717 shoppers):
last_ariel_size next_sub_brand     next_size  shoppers  share_%
           本体通常       ｱﾀｯｸ抗菌EX          本体通常     24663     11.0
           本体通常       ｱﾀｯｸ抗菌EX 詰替超ｳﾙﾄﾗｼﾞｬﾝﾎﾞ     22420     10.0
           本体通常       ｱﾀｯｸ抗菌EX  詰替ｳﾙﾄﾗｼﾞｬﾝﾎﾞ     21861      9.8
           本体通常     ﾎﾞｰﾙﾄﾞｼﾞｪﾙ          本体通常     16787      7.5
           本体通常       ｱﾀｯｸ抗菌EX         詰替超特大     14412      6.4
           本体通常 ﾎﾞｰﾙﾄﾞｼﾞｪﾙﾎﾞｰﾙ          本体通常     11211      5.0
           本体通常     ﾎﾞｰﾙﾄﾞｼﾞｪﾙ         詰替超特大      7385      3.3
           本体通常  ﾆｭｰﾋﾞｰｽﾞ ｼﾞｪﾙ         詰替超特大      7124      3.2
           本体通常            ｿﾉﾀ           ｿﾉﾀ      5190      2.3
           本体通常    ﾆｭｰﾋﾞｰｽﾞ 粉末          本体通常      4084      1.8

▶ From 詰替超特大 (258,583 shoppers):
last_ariel_size next_sub_brand     next_size  shoppers  share_%
          詰替超特大       ｱﾀｯｸ抗菌EX         詰替超特大     53708     20.8
          詰替超特大       ｱﾀｯｸ抗菌EX 詰替超ｳﾙﾄﾗｼﾞｬ

In [9]:
# ── Treemap: post-lapse destination ───────────────────────────────────
# Add category exit as a row
tree_data = dest_summary.copy()
tree_data = pd.concat([tree_data, pd.DataFrame([{
    'next_sub_brand': '(Category Exit)',
    'next_size': 'No Purchase',
    'shoppers': category_exit,
    'share_%': round(category_exit/total_lapsed*100, 1)
}])], ignore_index=True)

tree_data['label'] = tree_data['next_sub_brand'] + ' | ' + tree_data['next_size']

fig = px.treemap(tree_data, path=['next_sub_brand', 'next_size'], values='shoppers',
                 title='Post-Lapse Destination: Where Do Lost Ariel Shoppers Go?',
                 color='shoppers', color_continuous_scale='Reds')
fig.update_layout(height=600)
fig.show()

### Per-Size Destination Treemaps
Where do lapsed shoppers from each specific Ariel size go?

In [10]:
# ── Per-Size Destination Treemaps ──────────────────────────────────────
import plotly.express as px

ariel_sizes_lapse = order_and_filter_sizes(list(flow_detail['last_ariel_size'].unique()))

for ariel_size in ariel_sizes_lapse:
    size_flow = flow_detail[flow_detail['last_ariel_size'] == ariel_size].copy()
    total_from_size = size_flow['shoppers'].sum()

    if total_from_size < 10:
        print(f'Skipping {ariel_size}: too few observations ({total_from_size})')
        continue

    # Count category exit for this size
    lapsed_from_size = df_lapsed[df_lapsed['last_ariel_size'] == ariel_size]['shopper_key'].nunique()
    tracked_from_size = total_from_size
    exit_from_size = lapsed_from_size - tracked_from_size

    # Build treemap data
    td = size_flow[['next_sub_brand', 'next_size', 'shoppers']].copy()
    if exit_from_size > 0:
        td = pd.concat([td, pd.DataFrame([{
            'next_sub_brand': '(Category Exit)',
            'next_size': 'No Purchase',
            'shoppers': exit_from_size
        }])], ignore_index=True)

    td['share_%'] = (td['shoppers'] / lapsed_from_size * 100).round(1)

    fig = px.treemap(
        td, path=['next_sub_brand', 'next_size'], values='shoppers',
        title=f'From Ariel {ariel_size}: Where Do Lapsed Shoppers Go? (n={lapsed_from_size:,})',
        color='shoppers', color_continuous_scale='Reds'
    )
    fig.update_layout(height=550)
    fig.show()

    # Print top destinations
    print(f'\n▶ Ariel {ariel_size} → Top 5 Destinations:')
    top5 = td.sort_values('shoppers', ascending=False).head(5)
    for _, r in top5.iterrows():
        print(f'   {r["next_sub_brand"]:25s} {r["next_size"]:20s} {int(r["shoppers"]):>8,} ({r["share_%"]:.1f}%)')


▶ Ariel 本体通常 → Top 5 Destinations:
   (Category Exit)           No Purchase           182,545 (44.9%)
   ｱﾀｯｸ抗菌EX                  本体通常                   24,663 (6.1%)
   ｱﾀｯｸ抗菌EX                  詰替超ｳﾙﾄﾗｼﾞｬﾝﾎﾞ          22,420 (5.5%)
   ｱﾀｯｸ抗菌EX                  詰替ｳﾙﾄﾗｼﾞｬﾝﾎﾞ           21,861 (5.4%)
   ﾎﾞｰﾙﾄﾞｼﾞｪﾙ                本体通常                   16,787 (4.1%)



▶ Ariel 詰替超特大 → Top 5 Destinations:
   (Category Exit)           No Purchase           235,457 (47.7%)
   ｱﾀｯｸ抗菌EX                  詰替超特大                  53,708 (10.9%)
   ｱﾀｯｸ抗菌EX                  詰替超ｳﾙﾄﾗｼﾞｬﾝﾎﾞ          27,807 (5.6%)
   ｱﾀｯｸ抗菌EX                  詰替ｳﾙﾄﾗｼﾞｬﾝﾎﾞ           27,543 (5.6%)
   ﾆｭｰﾋﾞｰｽﾞ ｼﾞｪﾙ             詰替超特大                  12,568 (2.5%)



▶ Ariel 詰替ｳﾙﾄﾗｼﾞｬﾝﾎﾞ → Top 5 Destinations:
   (Category Exit)           No Purchase           165,784 (51.5%)
   ｱﾀｯｸ抗菌EX                  詰替超ｳﾙﾄﾗｼﾞｬﾝﾎﾞ          26,813 (8.3%)
   ｱﾀｯｸ抗菌EX                  詰替ｳﾙﾄﾗｼﾞｬﾝﾎﾞ           18,049 (5.6%)
   ｱﾀｯｸ抗菌EX                  詰替ﾒｶﾞｼﾞｬﾝﾎﾞ            14,156 (4.4%)
   ｱﾀｯｸ抗菌EX                  詰替超特大                  13,735 (4.3%)



▶ Ariel 詰替超ｳﾙﾄﾗｼﾞｬﾝﾎﾞ → Top 5 Destinations:
   (Category Exit)           No Purchase           215,243 (54.9%)
   ｱﾀｯｸ抗菌EX                  詰替超ｳﾙﾄﾗｼﾞｬﾝﾎﾞ          64,516 (16.4%)
   ｱﾀｯｸ抗菌EX                  詰替ｳﾙﾄﾗｼﾞｬﾝﾎﾞ           14,663 (3.7%)
   ｴﾏｰﾙ                      詰替超特大                   6,417 (1.6%)
   ｱﾀｯｸ抗菌EX                  詰替超特大                   5,975 (1.5%)



▶ Ariel 詰替ﾒｶﾞｼﾞｬﾝﾎﾞ → Top 5 Destinations:
   (Category Exit)           No Purchase            93,682 (53.4%)
   ｱﾀｯｸ抗菌EX                  詰替ﾒｶﾞｼﾞｬﾝﾎﾞ            43,165 (24.6%)
   CAINZ                     ｿﾉﾀ                     5,505 (3.1%)
   ﾆｭｰﾋﾞｰｽﾞ ｼﾞｪﾙ             詰替ﾒｶﾞｼﾞｬﾝﾎﾞ             2,577 (1.5%)
   ﾎﾞｰﾙﾄﾞｼﾞｪﾙ                詰替ﾒｶﾞｼﾞｬﾝﾎﾞ             2,559 (1.5%)


In [11]:
# ── Focus: Ariel → Attack flow by size ────────────────────────────────
attack_dest = flow_detail[flow_detail['next_sub_brand'] == ATTACK_EX].copy()

if len(attack_dest) > 0:
    print('=' * 60)
    print('CRITICAL: Lapse to アタック抗菌EX — by Size Flow')
    print('=' * 60)

    for size in order_and_filter_sizes(list(attack_dest['last_ariel_size'].unique())):
        s = attack_dest[attack_dest['last_ariel_size'] == size]
        total = s['shoppers'].sum()
        s_pct = s.copy()
        s_pct['pct'] = (s_pct['shoppers'] / total * 100).round(1)
        print(f'\n  From Ariel {size} → Attack sizes:')
        print(s_pct[['next_size', 'shoppers', 'pct']].to_string(index=False))
else:
    print('No lapse flow to Attack detected in data')

CRITICAL: Lapse to アタック抗菌EX — by Size Flow

  From Ariel 本体通常 → Attack sizes:
    next_size  shoppers  pct
         本体通常     24663 29.1
詰替超ｳﾙﾄﾗｼﾞｬﾝﾎﾞ     22420 26.4
 詰替ｳﾙﾄﾗｼﾞｬﾝﾎﾞ     21861 25.8
        詰替超特大     14412 17.0
  詰替ﾒｶﾞｼﾞｬﾝﾎﾞ      1418  1.7
          ｿﾉﾀ        20  0.0

  From Ariel 詰替超特大 → Attack sizes:
    next_size  shoppers  pct
        詰替超特大     53708 45.0
詰替超ｳﾙﾄﾗｼﾞｬﾝﾎﾞ     27807 23.3
 詰替ｳﾙﾄﾗｼﾞｬﾝﾎﾞ     27543 23.1
         本体通常     10206  8.5
  詰替ﾒｶﾞｼﾞｬﾝﾎﾞ       163  0.1
          ｿﾉﾀ        20  0.0

  From Ariel 詰替ｳﾙﾄﾗｼﾞｬﾝﾎﾞ → Attack sizes:
    next_size  shoppers  pct
詰替超ｳﾙﾄﾗｼﾞｬﾝﾎﾞ     26813 35.1
 詰替ｳﾙﾄﾗｼﾞｬﾝﾎﾞ     18049 23.6
  詰替ﾒｶﾞｼﾞｬﾝﾎﾞ     14156 18.5
        詰替超特大     13735 18.0
         本体通常      3611  4.7
          ｿﾉﾀ       135  0.2

  From Ariel 詰替超ｳﾙﾄﾗｼﾞｬﾝﾎﾞ → Attack sizes:
    next_size  shoppers  pct
詰替超ｳﾙﾄﾗｼﾞｬﾝﾎﾞ     64516 72.5
 詰替ｳﾙﾄﾗｼﾞｬﾝﾎﾞ     14663 16.5
        詰替超特大      5975  6.7
         本体通常      3795  4.3
          ｿﾉﾀ         8  0.0

  From Ariel 詰替

In [12]:
# ── Export Phase 4 results ────────────────────────────────────────────
output_file = 'phase4_lapse_analysis.xlsx'

# Strip timezone for Excel compatibility
def strip_tz(df):
    df = df.copy()
    for col in df.select_dtypes(include=['datetimetz']).columns:
        df[col] = df[col].dt.tz_localize(None)
    for col in df.columns:
        if df[col].dtype == 'object':
            try:
                df[col] = df[col].astype(str)
            except:
                pass
    return df

with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
    strip_tz(lapse_rate).to_excel(writer, sheet_name='Lapse_Rate_by_Size', index=False)
    strip_tz(asp_lapse).to_excel(writer, sheet_name='Lapse_by_ASP_Band', index=False)
    strip_tz(asp_lapse_all).to_excel(writer, sheet_name='Lapse_ASP_AllSizes', index=False)
    strip_tz(dest_summary).to_excel(writer, sheet_name='Destination_Summary', index=False)
    strip_tz(flow_detail).to_excel(writer, sheet_name='Flow_Detail', index=False)
    if len(attack_dest) > 0:
        strip_tz(attack_dest).to_excel(writer, sheet_name='Ariel_to_Attack', index=False)

print(f'✅ Phase 4 results exported to {output_file}')
print(f'   Sheets: Lapse_Rate_by_Size, Lapse_by_ASP_Band, Lapse_ASP_AllSizes, Destination_Summary, Flow_Detail, Ariel_to_Attack')

✅ Phase 4 results exported to phase4_lapse_analysis.xlsx
   Sheets: Lapse_Rate_by_Size, Lapse_by_ASP_Band, Lapse_ASP_AllSizes, Destination_Summary, Flow_Detail, Ariel_to_Attack


In [13]:
# ── (NEW) Step 5: アタック抗菌EX Lapse Analysis ────────────────────────
# Mirror of the Ariel lapse analysis but for Attack 抗菌EX shoppers
# Identifies Attack shoppers who lapsed, their ASP, and where they went

attack_lapse_query = f"""
WITH attack_purchases AS (
    SELECT
        idpos.shopper_key,
        prod.jp_segment_4_name AS size_code,
        CAST(idpos.sales_period_group_end_date_part AS DATE) AS purchase_date,
        SUM(idpos.pos_sales_amt) / SUM(idpos.pos_unit_sales_qty) AS asp
    FROM cdl_customer_prod.gold_customer_loyalty.loyalty_transact_fct_v1_vw idpos
    LEFT JOIN id_pos_ai_1.prod_dim_ext_vw prod
           ON idpos.prod_key = prod.prod_key
    LEFT JOIN id_pos_ai_1.shopper_dim_generic_vw shopper
           ON idpos.shopper_key = shopper.shopper_key
    WHERE idpos.sales_period_group_end_date_part BETWEEN '{ANALYSIS_START}' AND '{ANALYSIS_END}'
      AND idpos.data_provider_code_part IN ({RETAILER_IN})
      AND prod.jp_category_name = '{CATEGORY}'
      AND prod.jp_sub_category_alter_lang_name = '{SUB_CAT}'
      AND prod.jp_sub_brand_alter_lang_name = '{ATTACK_EX}'
      AND shopper.member_ind = 'Y'
      AND idpos.pos_unit_sales_qty > 0
    GROUP BY 1, 2, 3
),
last_attack AS (
    SELECT
        shopper_key,
        MAX(purchase_date) AS last_attack_date
    FROM attack_purchases
    GROUP BY 1
    HAVING MAX(purchase_date) <= '{LAPSE_CUTOFF_DATE}'
),
lapse_check AS (
    SELECT
        la.shopper_key,
        la.last_attack_date,
        MAX(CASE WHEN ap2.purchase_date > la.last_attack_date
                  AND ap2.purchase_date <= DATE_ADD(la.last_attack_date, {LAPSE_WINDOW_DAYS})
                 THEN 1 ELSE 0 END) AS returned
    FROM last_attack la
    LEFT JOIN attack_purchases ap2
           ON la.shopper_key = ap2.shopper_key
          AND ap2.purchase_date > la.last_attack_date
    GROUP BY 1, 2
),
lapsed_shoppers AS (
    SELECT shopper_key, last_attack_date
    FROM lapse_check
    WHERE returned = 0
)
SELECT
    ls.shopper_key,
    ls.last_attack_date,
    ap.size_code AS last_attack_size,
    ap.asp       AS last_attack_asp
FROM lapsed_shoppers ls
INNER JOIN attack_purchases ap
       ON ls.shopper_key     = ap.shopper_key
      AND ls.last_attack_date = ap.purchase_date
ORDER BY ls.last_attack_date
"""

print('⏳ Identifying lapsed Attack 抗菌EX shoppers...', flush=True)
df_lapsed_attack = execute_query(attack_lapse_query)
df_lapsed_attack['last_attack_date'] = pd.to_datetime(df_lapsed_attack['last_attack_date'])
df_lapsed_attack['last_attack_asp']  = pd.to_numeric(df_lapsed_attack['last_attack_asp'])

# Lapse rate by size
attack_active_query = f"""
SELECT prod.jp_segment_4_name AS size_code,
       COUNT(DISTINCT idpos.shopper_key) AS active_shoppers
FROM cdl_customer_prod.gold_customer_loyalty.loyalty_transact_fct_v1_vw idpos
LEFT JOIN id_pos_ai_1.prod_dim_ext_vw prod ON idpos.prod_key = prod.prod_key
LEFT JOIN id_pos_ai_1.shopper_dim_generic_vw shopper ON idpos.shopper_key = shopper.shopper_key
WHERE idpos.sales_period_group_end_date_part BETWEEN '{ANALYSIS_START}' AND '{LAPSE_CUTOFF_DATE}'
  AND idpos.data_provider_code_part IN ({RETAILER_IN})
  AND prod.jp_category_name = '{CATEGORY}'
  AND prod.jp_sub_category_alter_lang_name = '{SUB_CAT}'
  AND prod.jp_sub_brand_alter_lang_name = '{ATTACK_EX}'
  AND shopper.member_ind = 'Y'
  AND idpos.pos_unit_sales_qty > 0
GROUP BY 1
"""
df_attack_active = execute_query(attack_active_query)
df_attack_active['active_shoppers'] = pd.to_numeric(df_attack_active['active_shoppers'])
df_attack_active = df_attack_active[~df_attack_active['size_code'].isin(EXCLUDED_SIZES)]

atk_lapse_by_size = df_lapsed_attack[
    ~df_lapsed_attack['last_attack_size'].isin(EXCLUDED_SIZES)
].groupby('last_attack_size').agg(
    lapsed_shoppers=('shopper_key', 'nunique')
).reset_index().rename(columns={'last_attack_size': 'size_code'})

atk_lapse_rate = atk_lapse_by_size.merge(df_attack_active, on='size_code', how='left')
atk_lapse_rate['lapse_rate_%'] = (
    atk_lapse_rate['lapsed_shoppers'] / atk_lapse_rate['active_shoppers'] * 100
).round(1)
atk_lapse_rate['_sort'] = atk_lapse_rate['size_code'].map(
    {s: i for i, s in enumerate(SIZE_ORDER)}
).fillna(99)
atk_lapse_rate = atk_lapse_rate.sort_values('_sort').drop(columns='_sort')

print(f'\n✅ {len(df_lapsed_attack):,} lapsed Attack shoppers identified')
print('\n' + '=' * 60)
print('Lapse Rate by Size — アタック抗菌EX')
print('=' * 60)
print(atk_lapse_rate.to_string(index=False))


⏳ Identifying lapsed Attack 抗菌EX shoppers...

✅ 2,551,031 lapsed Attack shoppers identified

Lapse Rate by Size — アタック抗菌EX
    size_code  lapsed_shoppers  active_shoppers  lapse_rate_%
         本体通常           173577           407299          42.6
        詰替超特大           656405          1560321          42.1
 詰替ｳﾙﾄﾗｼﾞｬﾝﾎﾞ           489967          1446373          33.9
詰替超ｳﾙﾄﾗｼﾞｬﾝﾎﾞ           966401          2302661          42.0
  詰替ﾒｶﾞｼﾞｬﾝﾎﾞ           259726           570104          45.6


In [14]:
# ── (NEW) Attack Post-Lapse Destination ───────────────────────────────
attack_dest_query = f"""
WITH attack_last AS (
    SELECT
        idpos.shopper_key,
        MAX(CAST(idpos.sales_period_group_end_date_part AS DATE)) AS last_attack_date
    FROM cdl_customer_prod.gold_customer_loyalty.loyalty_transact_fct_v1_vw idpos
    INNER JOIN id_pos_ai_1.prod_dim_ext_vw prod ON idpos.prod_key = prod.prod_key
    INNER JOIN id_pos_ai_1.shopper_dim_generic_vw shopper ON idpos.shopper_key = shopper.shopper_key
    WHERE idpos.sales_period_group_end_date_part BETWEEN '{ANALYSIS_START}' AND '{ANALYSIS_END}'
      AND idpos.data_provider_code_part IN ({RETAILER_IN})
      AND prod.jp_category_name = '{CATEGORY}'
      AND prod.jp_sub_category_alter_lang_name = '{SUB_CAT}'
      AND prod.jp_sub_brand_alter_lang_name = '{ATTACK_EX}'
      AND shopper.member_ind = 'Y'
      AND idpos.pos_unit_sales_qty > 0
    GROUP BY 1
    HAVING MAX(CAST(idpos.sales_period_group_end_date_part AS DATE)) <= DATE('{LAPSE_CUTOFF_DATE}')
),
next_laundry AS (
    SELECT
        al.shopper_key,
        prod.jp_sub_brand_alter_lang_name AS next_sub_brand,
        prod.jp_segment_4_name            AS next_size,
        ROW_NUMBER() OVER (
            PARTITION BY al.shopper_key
            ORDER BY CAST(idpos.sales_period_group_end_date_part AS DATE)
        ) AS rn
    FROM attack_last al
    INNER JOIN cdl_customer_prod.gold_customer_loyalty.loyalty_transact_fct_v1_vw idpos
           ON al.shopper_key = idpos.shopper_key
    INNER JOIN id_pos_ai_1.prod_dim_ext_vw prod ON idpos.prod_key = prod.prod_key
    WHERE CAST(idpos.sales_period_group_end_date_part AS DATE) > al.last_attack_date
      AND idpos.sales_period_group_end_date_part <= '{ANALYSIS_END}'
      AND idpos.data_provider_code_part IN ({RETAILER_IN})
      AND prod.jp_category_name = '{CATEGORY}'
      AND prod.jp_sub_category_alter_lang_name = '{SUB_CAT}'
      AND idpos.pos_unit_sales_qty > 0
)
SELECT shopper_key, next_sub_brand, next_size
FROM next_laundry
WHERE rn = 1
"""

print('⏳ Tracking Attack post-lapse destinations...', flush=True)
df_attack_destination = execute_query(attack_dest_query)
df_attack_dest_full = df_attack_destination.merge(
    df_lapsed_attack[['shopper_key', 'last_attack_size', 'last_attack_asp']],
    on='shopper_key', how='left'
)

atk_dest_summary = df_attack_destination.groupby(['next_sub_brand', 'next_size']).agg(
    shoppers=('shopper_key', 'nunique')
).reset_index().sort_values('shoppers', ascending=False)

total_atk_lapsed   = df_lapsed_attack['shopper_key'].nunique()
total_atk_tracked  = atk_dest_summary['shoppers'].sum()
atk_exit           = total_atk_lapsed - total_atk_tracked

print(f'\n✅ Attack lapsed: {total_atk_lapsed:,} | Tracked: {total_atk_tracked:,} | Category exit: {atk_exit:,}')
print('\nTop Destinations from Attack Lapse:')
atk_dest_summary['share_%'] = (atk_dest_summary['shoppers'] / total_atk_lapsed * 100).round(1)
print(atk_dest_summary.head(20).to_string(index=False))

# ── Per-size destination from Attack lapse ─────────────────────────────
atk_flow_detail = df_attack_dest_full.groupby(['last_attack_size', 'next_sub_brand', 'next_size']).agg(
    shoppers=('shopper_key', 'nunique')
).reset_index().sort_values(['last_attack_size', 'shoppers'], ascending=[True, False])

print('\nAttack Lapse Per-Size Treemaps:')
atk_sizes_lapse = order_and_filter_sizes(
    df_attack_dest_full['last_attack_size'].dropna().unique()
)

for atk_size in atk_sizes_lapse:
    size_flow = atk_flow_detail[atk_flow_detail['last_attack_size'] == atk_size].copy()
    total_from_size = size_flow['shoppers'].sum()
    if total_from_size < 10:
        continue

    lapsed_from_size = df_lapsed_attack[
        df_lapsed_attack['last_attack_size'] == atk_size
    ]['shopper_key'].nunique()
    exit_from_size = lapsed_from_size - total_from_size

    td = size_flow[['next_sub_brand', 'next_size', 'shoppers']].copy()
    if exit_from_size > 0:
        td = pd.concat([td, pd.DataFrame([{
            'next_sub_brand': '(Category Exit)',
            'next_size': 'No Purchase',
            'shoppers': exit_from_size
        }])], ignore_index=True)

    td['share_%'] = (td['shoppers'] / lapsed_from_size * 100).round(1)

    fig = px.treemap(
        td, path=['next_sub_brand', 'next_size'], values='shoppers',
        title=f'From Attack {atk_size}: Where Do Lapsed Shoppers Go? (n={lapsed_from_size:,})',
        color='shoppers', color_continuous_scale='Oranges'
    )
    fig.update_layout(height=550)
    fig.show()

    print(f'\n▶ Attack {atk_size} → Top 5 Destinations:')
    top5 = td.sort_values('shoppers', ascending=False).head(5)
    for _, r in top5.iterrows():
        print(f'   {r["next_sub_brand"]:25s} {r["next_size"]:20s} {int(r["shoppers"]):>8,} ({r["share_%"]:.1f}%)')

# ── Focus: Attack → Ariel flow ─────────────────────────────────────────
ariel_from_attack = atk_flow_detail[atk_flow_detail['next_sub_brand'] == ARIEL_GEL].copy()
if len(ariel_from_attack) > 0:
    print('\n' + '=' * 60)
    print('OPPORTUNITY: Attack Lapses → Switching to アリエールジェル')
    print('=' * 60)
    for size in order_and_filter_sizes(ariel_from_attack['last_attack_size'].unique()):
        s = ariel_from_attack[ariel_from_attack['last_attack_size'] == size]
        total = s['shoppers'].sum()
        s_pct = s.assign(pct=(s['shoppers'] / total * 100).round(1))
        print(f'\n  From Attack {size} → Ariel sizes:')
        print(s_pct[['next_size', 'shoppers', 'pct']].to_string(index=False))
else:
    print('No Attack lapse → Ariel flow in data')


⏳ Tracking Attack post-lapse destinations...

✅ Attack lapsed: 2,513,563 | Tracked: 932,264 | Category exit: 1,581,299

Top Destinations from Attack Lapse:
next_sub_brand     next_size  shoppers  share_%
     ｱﾘｴｰﾙｼﾞｪﾙ         詰替超特大     84378      3.4
     ｱﾘｴｰﾙｼﾞｪﾙ 詰替超ｳﾙﾄﾗｼﾞｬﾝﾎﾞ     77866      3.1
 ﾆｭｰﾋﾞｰｽﾞ ｼﾞｪﾙ         詰替超特大     39408      1.6
     ｱﾘｴｰﾙｼﾞｪﾙ          本体通常     38007      1.5
          ｴﾏｰﾙ         詰替超特大     35947      1.4
     ｱﾘｴｰﾙｼﾞｪﾙ  詰替ｳﾙﾄﾗｼﾞｬﾝﾎﾞ     34897      1.4
      ｱﾀｯｸZERO     詰替超ｼﾞｬﾝﾎﾞ     31195      1.2
          ｱｸﾛﾝ         詰替超特大     25805      1.0
 ﾆｭｰﾋﾞｰｽﾞ ｼﾞｪﾙ  詰替ｳﾙﾄﾗｼﾞｬﾝﾎﾞ     25527      1.0
        ｱﾀｯｸ粉末          詰替通常     23999      1.0
     ｱﾘｴｰﾙｼﾞｪﾙ   詰替ﾒｶﾞｼﾞｬﾝﾎﾞ     23729      0.9
      ｱﾀｯｸZERO  詰替ｳﾙﾄﾗｼﾞｬﾝﾎﾞ     23132      0.9
    ﾎﾞｰﾙﾄﾞｼﾞｪﾙ         詰替超特大     19626      0.8
   ﾆｭｰﾋﾞｰｽﾞ 粉末          本体通常     19585      0.8
ﾎﾞｰﾙﾄﾞｼﾞｪﾙﾎﾞｰﾙ          本体通常     16482      0.7
           ｿﾉﾀ           ｿﾉﾀ     16470      0.7
ﾎﾞｰﾙﾄﾞｼﾞｪﾙﾎﾞｰﾙ 詰替ﾊｲﾊﾟｰｼﾞｬﾝﾎﾞ


▶ Attack 本体通常 → Top 5 Destinations:
   (Category Exit)           No Purchase            92,767 (53.4%)
   ｱﾘｴｰﾙｼﾞｪﾙ                 本体通常                    6,211 (3.6%)
   ｱﾘｴｰﾙｼﾞｪﾙ                 詰替超特大                   5,215 (3.0%)
   ｱﾘｴｰﾙｼﾞｪﾙ                 詰替超ｳﾙﾄﾗｼﾞｬﾝﾎﾞ           2,893 (1.7%)
   ﾅﾉｯｸｽﾜﾝ                   本体大                     2,712 (1.6%)



▶ Attack 詰替超特大 → Top 5 Destinations:
   (Category Exit)           No Purchase           390,305 (59.5%)
   ｱﾘｴｰﾙｼﾞｪﾙ                 詰替超特大                  50,527 (7.7%)
   ﾆｭｰﾋﾞｰｽﾞ ｼﾞｪﾙ             詰替超特大                  24,837 (3.8%)
   ｱﾘｴｰﾙｼﾞｪﾙ                 本体通常                   12,220 (1.9%)
   ｱﾘｴｰﾙｼﾞｪﾙ                 詰替ｳﾙﾄﾗｼﾞｬﾝﾎﾞ           10,492 (1.6%)



▶ Attack 詰替ｳﾙﾄﾗｼﾞｬﾝﾎﾞ → Top 5 Destinations:
   (Category Exit)           No Purchase           302,181 (61.7%)
   ｱﾘｴｰﾙｼﾞｪﾙ                 詰替超特大                  16,331 (3.3%)
   ｱﾘｴｰﾙｼﾞｪﾙ                 詰替超ｳﾙﾄﾗｼﾞｬﾝﾎﾞ          14,813 (3.0%)
   ｱﾘｴｰﾙｼﾞｪﾙ                 詰替ｳﾙﾄﾗｼﾞｬﾝﾎﾞ           11,758 (2.4%)
   ｴﾏｰﾙ                      詰替超特大                   8,854 (1.8%)



▶ Attack 詰替超ｳﾙﾄﾗｼﾞｬﾝﾎﾞ → Top 5 Destinations:
   (Category Exit)           No Purchase           641,414 (66.4%)
   ｱﾘｴｰﾙｼﾞｪﾙ                 詰替超ｳﾙﾄﾗｼﾞｬﾝﾎﾞ          52,190 (5.4%)
   ｴﾏｰﾙ                      詰替超特大                  16,070 (1.7%)
   ｱﾀｯｸZERO                  詰替ｳﾙﾄﾗｼﾞｬﾝﾎﾞ           13,945 (1.4%)
   ｱﾘｴｰﾙｼﾞｪﾙ                 本体通常                   12,922 (1.3%)



▶ Attack 詰替ﾒｶﾞｼﾞｬﾝﾎﾞ → Top 5 Destinations:
   (Category Exit)           No Purchase           174,216 (67.1%)
   ｱﾘｴｰﾙｼﾞｪﾙ                 詰替ﾒｶﾞｼﾞｬﾝﾎﾞ            21,501 (8.3%)
   CAINZ                     ｿﾉﾀ                    10,702 (4.1%)
   ｱﾀｯｸZERO                  詰替超ｳﾙﾄﾗｼﾞｬﾝﾎﾞ           5,973 (2.3%)
   ﾆｭｰﾋﾞｰｽﾞ ｼﾞｪﾙ             詰替ﾒｶﾞｼﾞｬﾝﾎﾞ             5,687 (2.2%)

OPPORTUNITY: Attack Lapses → Switching to アリエールジェル

  From Attack 本体通常 → Ariel sizes:
    next_size  shoppers  pct
         本体通常      6211 36.1
        詰替超特大      5215 30.3
詰替超ｳﾙﾄﾗｼﾞｬﾝﾎﾞ      2893 16.8
 詰替ｳﾙﾄﾗｼﾞｬﾝﾎﾞ      1507  8.8
  詰替ﾒｶﾞｼﾞｬﾝﾎﾞ      1201  7.0
          ｿﾉﾀ       161  0.9
    詰替超ｼﾞｬﾝﾎﾞ        14  0.1
         詰替通常         1  0.0

  From Attack 詰替超特大 → Ariel sizes:
    next_size  shoppers  pct
        詰替超特大     50527 60.3
         本体通常     12220 14.6
 詰替ｳﾙﾄﾗｼﾞｬﾝﾎﾞ     10492 12.5
詰替超ｳﾙﾄﾗｼﾞｬﾝﾎﾞ      8986 10.7
  詰替ﾒｶﾞｼﾞｬﾝﾎﾞ      1204  1.4
          ｿﾉﾀ       302  0.4
    詰替超ｼﾞｬﾝﾎﾞ       121  0.1
         